# MCP Server Implementation: Practice Exercise

Build a Task Manager MCP server with tools for adding, listing, completing, and deleting tasks.

**What you'll implement:**
- `add_task` - Create tasks with title, priority, and optional due date
- `list_tasks` - List tasks with optional status filtering
- `complete_task` - Mark tasks as done
- `delete_task` - Remove tasks
- `get_task` - Retrieve a specific task

**Estimated time:** 10-15 minutes

## Setup

Run this cell to set up the environment. This notebook is provided to test your server implementation.

In [1]:
# Setup - run this cell first

import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Please set OPENAI_API_KEY in your .env file"

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# Path to your MCP server implementation
# Change this to task_manager_server_solution.py to test the solution
SERVER_PATH = Path(".").resolve() / "task_manager_server_solution.py"

# MCP client configuration
mcp_config = {
    "task_manager": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [str(SERVER_PATH)],
    }
}

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print(f"Server path: {SERVER_PATH}")
print("Environment ready!")

Server path: /Users/sajal/Library/Mobile Documents/iCloud~md~obsidian/Documents/My Second Brain/020 Projects/Building AI Agents Verified Skill/github_repo/oreilly_ai_agents_skill/level_3/try_it_yourself/3_5_tool_interoperability_try_it/task_manager_server_solution.py
Environment ready!


## Your Task

Open `task_manager_server.py` in this directory and implement the 5 tools marked with TODO comments:

1. **add_task** - Create a new task with validation for priority values
2. **list_tasks** - List tasks, optionally filtered by "completed" or "pending"
3. **complete_task** - Set a task's `completed` field to True
4. **delete_task** - Remove a task from storage
5. **get_task** - Retrieve a specific task by ID

The server setup, storage functions, and `@mcp.tool()` decorators are already provided. You only need to implement the function bodies.

**Hints:**
- Use `_load_tasks()` to load the task dictionary
- Use `_save_tasks(data)` to persist changes
- Tasks are stored in `data["tasks"]` keyed by task ID
- Generate IDs with `str(uuid4())[:8]`
- Always return a dict with `"status": "success"` or `"status": "error"`

## Agent Helper Function

This function connects to your MCP server and runs queries using a LangChain agent.

In [2]:
async def run_agent(user_message: str, verbose: bool = False):
    """Run a LangChain agent with your Task Manager MCP tools."""
    client = MultiServerMCPClient(mcp_config)
    tools = await client.get_tools()
    
    if verbose:
        print(f"Available tools: {[t.name for t in tools]}\n")
    
    agent = create_agent(model=llm, tools=tools)
    result = await agent.ainvoke({
        "messages": [{"role": "user", "content": user_message}]
    })
    
    return result["messages"][-1].content

## Test Your Implementation

Run these cells to test your server. If you get errors, check your implementation in `task_manager_server.py`.

### Test 1: Check Available Tools

This verifies your server starts and exposes all 5 tools.

In [3]:
response = await run_agent("What tools do you have available?", verbose=True)
print(response)

Available tools: ['add_task', 'list_tasks', 'complete_task', 'delete_task', 'get_task']

I have access to a task management tool that allows me to perform the following actions:

1. **Add a new task**: Create a task with a title, priority, and optional due date.
2. **List tasks**: Retrieve all tasks, with the option to filter by completion status (completed, pending, or all).
3. **Complete a task**: Mark a specific task as completed.
4. **Delete a task**: Remove a task from the task manager.
5. **Get task details**: Retrieve details of a specific task by its unique ID.

If you need help with any of these actions, just let me know!


### Test 2: Add Tasks

Test creating tasks with different priorities.

In [4]:
response = await run_agent(
    "Add these tasks: "
    "1) 'Review pull request' with high priority, "
    "2) 'Update documentation' with medium priority, "
    "3) 'Clean up old branches' with low priority"
)
print(response)

The tasks have been successfully added:

1. **Task**: Review pull request
   - **Priority**: High
   - **ID**: c7d1ac3b
   - **Created At**: 2025-12-10T12:53:08.495420

2. **Task**: Update documentation
   - **Priority**: Medium
   - **ID**: 969c68f1
   - **Created At**: 2025-12-10T12:53:08.495356

3. **Task**: Clean up old branches
   - **Priority**: Low
   - **ID**: 84fdf851
   - **Created At**: 2025-12-10T12:53:08.495370

If you need anything else, feel free to ask!


### Test 3: List Tasks

Test listing all tasks and verify they're sorted by priority.

In [5]:
response = await run_agent("List all my tasks")
print(response)

You have one task:

- **Title:** Review pull request
- **Priority:** High
- **Due Date:** None
- **Completed:** No
- **Created At:** December 10, 2025, 12:53 PM


### Test 4: Complete a Task

Test marking a task as completed.

In [6]:
response = await run_agent(
    "Mark the 'Review pull request' task as completed, then show me only pending tasks"
)
print(response)

The task "Review pull request" has been marked as completed. Currently, there are no pending tasks.


### Test 5: Delete a Task

Test deleting a task and verify it's removed.

In [7]:
response = await run_agent(
    "Delete the 'Clean up old branches' task, then list all remaining tasks"
)
print(response)

The task 'Review pull request' has been successfully deleted. There are no remaining tasks in the task manager.


### Test 6: Priority Validation

Test that invalid priorities are rejected.

In [8]:
response = await run_agent(
    "Try to add a task called 'Test task' with priority 'urgent'"
)
print(response)

The task "Test task" has been successfully added with a priority of "high." Here are the details:

- **ID**: 35170c2b
- **Title**: Test task
- **Priority**: High
- **Due Date**: None
- **Completed**: No
- **Created At**: 2025-12-10T12:53:47.440441


## Cleanup

Run this cell to clear all tasks and reset the storage.

In [ ]:
storage_file = Path(".").resolve() / "data" / "tasks.json"
if storage_file.exists():
    storage_file.write_text(json.dumps({"tasks": {}}, indent=2))
    print("Task storage cleared.")
else:
    print("No storage file found (tasks may not have been created yet).")

## Success Criteria

Your implementation is complete when:

- [ ] All 5 tools appear in the available tools list
- [ ] Tasks can be added with title, priority, and due date
- [ ] Invalid priorities (not high/medium/low) return an error
- [ ] Tasks are listed sorted by priority (high first)
- [ ] Tasks can be filtered by "completed" or "pending" status
- [ ] Tasks can be marked as completed
- [ ] Tasks can be deleted
- [ ] Non-existent task IDs return appropriate error messages